In [1]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import mne

import utils
from featuregen import DFG
import preprocessing
import reader

In [3]:
# Read the csv file containing the ERP data and the 
df = pd.read_csv(r"C:\Mon disque D\Gipsa\6- Schizophrenia diagnosis\dataset\dataset 1\ERPdata.csv")
demographic = pd.read_csv("C:/Mon disque D/Gipsa/6- Schizophrenia diagnosis/dataset/dataset 1/demographic.csv")
diagnosis_dict = dict(zip(demographic.subject, demographic[" group"]))  # 1 SZ 0 CTL
print("Diagnosis dict\n", diagnosis_dict)
channels = list(df.columns[2:-1])
print("\n Channels:\n", channels)

Diagnosis dict
 {1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0, 16: 0, 17: 0, 18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 1, 26: 1, 27: 1, 28: 1, 29: 1, 30: 1, 31: 1, 32: 1, 33: 1, 34: 1, 35: 1, 36: 1, 37: 1, 38: 1, 39: 1, 40: 1, 41: 1, 42: 1, 43: 1, 44: 1, 45: 1, 46: 1, 47: 1, 48: 1, 49: 1, 50: 1, 51: 1, 52: 1, 53: 1, 54: 1, 55: 1, 56: 1, 57: 1, 58: 1, 59: 0, 60: 0, 61: 0, 62: 0, 63: 0, 64: 0, 65: 0, 66: 0, 67: 1, 68: 1, 69: 1, 70: 1, 71: 1, 72: 1, 73: 1, 74: 1, 75: 1, 76: 1, 77: 1, 78: 1, 79: 1, 80: 1, 81: 1}

 Channels:
 ['Fz', 'FCz', 'Cz', 'FC3', 'FC4', 'C3', 'C4', 'CP3', 'CP4']


In [4]:
df

,subject,condition,Fz,FCz,Cz,FC3,FC4,C3,C4,CP3,CP4,time_ms
0,1,1,5.533701,5.726507,5.469535,5.386723,4.588875,6.560092,4.542811,5.397492,5.103695,-1500.0000
1,1,1,5.651489,5.837326,5.773131,5.627975,4.822217,6.739976,4.811770,5.541357,5.379273,-1499.0234
2,1,1,5.717580,5.932924,5.948466,5.826460,4.979647,7.026199,5.053779,5.634972,5.600504,-1498.0469
3,1,1,5.703267,5.968103,5.851512,5.812192,4.992899,6.940671,5.106650,5.543577,5.589775,-1497.0703
4,1,1,5.571578,5.917541,5.812808,5.744715,4.963338,6.726491,5.158073,5.454069,5.614092,-1496.0938
...,...,...,...,...,...,...,...,...,...,...,...,...
746491,81,3,-0.401267,0.041014,-0.352556,0.712530,-0.427019,0.479170,1.041864,0.645761,-0.085649,1495.1172
746492,81,3,-0.440294,0.093863,-0.422151,0.792209,-0.469230,0.486767,0.955658,0.601938,-0.264824,1496.0938
746493,81,3,-0.466162,0.083799,-0.485091,0.799034,-0.466002,0.516740,0.972635,0.612470,-0.463196,1497.0703
746494,81,3,-0.472620,0.003017,-0.465663,0.675452,-0.408777,0.558901,0.943028,0.635287,-0.656748,1498.0469


In [3]:
# Verbosity
mne.set_log_level(verbose='WARNING')  # 'DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL'
reader.reader_verbose = False
preprocessing.preprocessing_verbose = True

# Stimuli management
stim_types = ['1', '2', '3']
features_container = dict([(stim, {}) for stim in stim_types])

In [4]:
# Parameters
fs = 1024
t_min = -0
t_max = 500 - 1 / fs
n_point = round((t_max - t_min) / 1000 * fs)
n_channel = len(channels)
print(f'{n_point = }, {channels = }, {n_channel = }')

# Class Parameters
my_reader = reader.Reader()

preprocess = preprocessing.PreProcessing(save=False, save_precision='double', overwrite=True,
                                         shift=False, t_shift=0.0,
                                         sampling_freq=None,
                                         filter=[None, 30], n_jobs=-1,
                                         rejection_type=None, reject_t_min=-0.1, reject_t_max=0.6,
                                         ref_channel=None,
                                         crop=True, crop_t_min=t_min, crop_t_max=t_max, include_t_max=True,
                                         baseline=[None, None])

feature_gen = DFG(method='LARS',
                  f_sampling=fs,
                  version=1, disable=None,  # 0
                  normalize=True,
                  model_freq=list(np.concatenate((np.linspace(1, 15, 20, endpoint=False), np.linspace(15, 30, 20)))),
                  damping=None,  # (under-damped 0.008 / over-damped 0.09)
                  alpha=8e-3,  # 8e-4  # for ICA removal do this 1e-5
                  fit_path=True, ols_fit=True,
                  fast=True,
                  selection=np.arange(0.05, 1.05, 0.05),
                  selection_alpha=None,
                  plot=False, show=True, fig_name="fig name", save_fig=False)

n_point = 512, channels = ['Fz', 'FCz', 'Cz', 'FC3', 'FC4', 'C3', 'C4', 'CP3', 'CP4'], n_channel = 9


In [5]:
# Session creation
date = datetime.now().strftime("%Y-%m-%d %H;%M")
session_folder = os.path.join(os.getcwd(), 'features', date)
backup_folder = os.path.join(os.getcwd(), 'features backup', date)

# Main
for subj in tqdm(df['subject'].unique()[[0]]):
    utils.print_c('\nReading file: {:}'.format(subj), bold=True)
    group = diagnosis_dict[subj]
    data = np.empty((3, 3072, n_channel))
    
    # data reading from CSV
    for stim_i, stim in enumerate(df['condition'].unique()):
        filt = (df['subject'] == subj) & (df['condition'] == stim)  # & (t_min <= df['time_ms']) & (df['time_ms'] <= t_max)
        data[stim_i, :, :] = df.loc[filt, 'Fz':'CP4']
    print(data.shape)
    # data parsing and pre-processing
    data_ = my_reader.data_to_mne(data, s_rate=fs, t_min=t_min, channels=channels, stim_types=stim_types,
                                  subj=subj, category=group)
    preprocess.set_data(data_, epoch_drop_idx=None, epoch_drop_reason='USER', channel_drop=None)
    data_ = preprocess.process()
    data = data_.get_data()
    print(data.shape)
    
    # Feature generation
    for stim_i, stim in enumerate(stim_types):
        features, x0 = feature_gen.generate(data[stim_i].T)
        features, x0 = utils.compress(features), utils.ndarray_to_list(x0)

        # Appending the new dynamical features to the dynamical feature container
        temp = {str(subj): {'features': features,
                            'x0': x0,
                            'subject_info': group}}
        features_container[stim].update(temp)
    plt.show()
    # here should be the call for each subject of the generator
utils.save_args(features_container, path=session_folder, save_name='generated_features', verbose=True)
utils.save_args(preprocess._saved_args, verbose=True, path=session_folder, save_name='preprocessing_parameters')
utils.save_args({**feature_gen.parameters, **{'channel_picks': channels}, **{'data_case': 'evoked'}}, 
                path=session_folder, save_name='DFG_parameters', verbose=True)

  0%|          | 0/1 [00:00<?, ?it/s]


Reading file: 1
	Preprocessing:
		No resampling performed
HEEEEY
		No eye blink rejection
		EEG data marked as already having the desired reference
		Cropping epochs between (0, 499.9990234375) [s]
		No baseline correction applied
(3, 9, 3072)
	Feature generation : LARS
		Version: 1
		alpha = 0.008
		Damping = 0
		Selection = [0.05 0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65 0.7
 0.75 0.8  0.85 0.9  0.95 1.  ]
		OLS fit: True
		Fast: True
		Model frequencies:
 [ 1.    1.7   2.4   3.1   3.8   4.5   5.2   5.9   6.6   7.3   8.    8.7
  9.4  10.1  10.8  11.5  12.2  12.9  13.6  14.3  15.   15.79 16.58 17.37
 18.16 18.95 19.74 20.53 21.32 22.11 22.89 23.68 24.47 25.26 26.05 26.84
 27.63 28.42 29.21 30.  ]


C:\Users\meghnouh\PycharmProjects\Schizophrenia Detection\preprocessing.py:271: RuntimeWarning: tmax is not in time interval. tmax is set to <class 'mne.epochs.EpochsArray'>.tmax (2.99902 sec)
  self._raw = self._raw.crop(tmin=self._crop_t_min, tmax=self._crop_t_max,
  0%|          | 0/1 [00:28<?, ?it/s]


KeyboardInterrupt: 